# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

optimizer = ModelOptimizer("ItemKNN_tversky")

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + '_tversky'

In [8]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "tversky",
        "topK": optuna_trial.suggest_int("topK", 5, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "tversky_alpha": optuna_trial.suggest_float("tversky_alpha", 0.0, 1.0),
        "tversky_beta": optuna_trial.suggest_float("tversky_beta", 0.0, 1.0),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["none", "TF-IDF", "BM25"]),
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 0.5, 2.0)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.0, 1.0)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-20 13:22:28,109] A new study created in RDB with name: ItemKNNCFRecommender_tversky


  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 4699.41 column/sec. Elapsed time 1.48 sec
  Fold 1/5 - Score: 0.18231095373630524
Similarity column 6969 (100.0%), 4733.27 column/sec. Elapsed time 1.47 sec
  Fold 2/5 - Score: 0.18310390412807465
Similarity column 6969 (100.0%), 4722.08 column/sec. Elapsed time 1.48 sec
  Fold 3/5 - Score: 0.1829211264848709
Similarity column 6969 (100.0%), 4762.87 column/sec. Elapsed time 1.46 sec
  Fold 4/5 - Score: 0.18212571740150452
Similarity column 6969 (100.0%), 4664.18 column/sec. Elapsed time 1.49 sec
  Fold 5/5 - Score: 0.18268044292926788
[I 2025-11-20 13:23:16,319] Trial 0 finished with value: 0.18262842297554016 and parameters: {'topK': 277, 'shrink': 724, 'tversky_alpha': 0.2820222255348235, 'tversky_beta': 0.6634491466165708, 'normalize': True, 'feature_weighting': 'TF-IDF'}. Best is trial 0 with value: 0.18262842297554016.
Similarity column 6969 (100.0%), 4715.03 column/sec. Elapsed time 1.48 sec
  Fold 1/5 - Score: 0.20542991161346436
Similarity colum

In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [13]:
optimizer.optimize(
    objective_function=objective_function,
    n_trials=40
)

  0%|          | 0/40 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 2656.88 column/sec. Elapsed time 2.62 sec
  Fold 1/5 - Score: 0.22277303040027618
Similarity column 6969 (100.0%), 3070.44 column/sec. Elapsed time 2.27 sec
  Fold 2/5 - Score: 0.22421573102474213
Similarity column 6969 (100.0%), 2127.51 column/sec. Elapsed time 3.28 sec
  Fold 3/5 - Score: 0.22413195669651031
Similarity column 6969 (100.0%), 3193.35 column/sec. Elapsed time 2.18 sec
  Fold 4/5 - Score: 0.2235424965620041
[I 2025-11-20 14:25:58,335] Trial 100 finished with value: 0.22366580367088318 and parameters: {'topK': 45, 'shrink': 148, 'tversky_alpha': 0.10425844570903713, 'tversky_beta': 0.8584427649586016, 'normalize': True, 'feature_weighting': 'TF-IDF'}. Best is trial 96 with value: 0.24625977873802185.
Similarity column 6969 (100.0%), 3575.27 column/sec. Elapsed time 1.95 sec
  Fold 1/5 - Score: 0.23814883828163147
Similarity column 6969 (100.0%), 3295.55 column/sec. Elapsed time 2.11 sec
  Fold 2/5 - Score: 0.23821023106575012
Similarity co

In [14]:
optuna.visualization.plot_optimization_history(optuna_study)

In [15]:
optuna.visualization.plot_param_importances(optuna_study)

In [16]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [18]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "tversky",
        "topK": optuna_trial.suggest_int("topK", 1, 10),
        "shrink": optuna_trial.suggest_int("shrink", 50, 70),
        "tversky_alpha": optuna_trial.suggest_float("tversky_alpha", 0.09, 0.1),
        "tversky_beta": optuna_trial.suggest_float("tversky_beta", 0.9, 1.0),
        "normalize": True,
        "feature_weighting": "none",
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [19]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

[I 2025-11-20 15:21:38,025] A new study created in RDB with name: ItemKNNCFRecommender_tversky_refined


  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 2696.28 column/sec. Elapsed time 2.58 sec
  Fold 1/5 - Score: 0.24534161388874054
Similarity column 6969 (100.0%), 3085.13 column/sec. Elapsed time 2.26 sec
  Fold 2/5 - Score: 0.24489383399486542
Similarity column 6969 (100.0%), 4029.83 column/sec. Elapsed time 1.73 sec
  Fold 3/5 - Score: 0.24538679420948029
Similarity column 6969 (100.0%), 3890.25 column/sec. Elapsed time 1.79 sec
  Fold 4/5 - Score: 0.24390065670013428
Similarity column 6969 (100.0%), 2819.73 column/sec. Elapsed time 2.47 sec
  Fold 5/5 - Score: 0.24608869850635529
[I 2025-11-20 15:22:15,160] Trial 0 finished with value: 0.24512231349945068 and parameters: {'topK': 4, 'shrink': 57, 'tversky_alpha': 0.09494913018583631, 'tversky_beta': 0.9171947791827617}. Best is trial 0 with value: 0.24512231349945068.
Similarity column 6969 (100.0%), 3560.39 column/sec. Elapsed time 1.96 sec
  Fold 1/5 - Score: 0.24798551201820374
Similarity column 6969 (100.0%), 3130.18 column/sec. Elapsed time 2

In [20]:
optuna.visualization.plot_optimization_history(optuna_study)

In [21]:
optuna.visualization.plot_param_importances(optuna_study)

In [22]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Trial 1
Best Value: 0.24763381481170654
Best Params: {'topK': 5, 'shrink': 62, 'tversky_alpha': 0.0913436157822328, 'tversky_beta': 0.9357058146064209, 'normalize': True, 'feature_weighting': 'none'}

- Trial 2
Best Value: 0.24964454770088196
Best Params: {'topK': 6, 'shrink': 61, 'tversky_alpha': 0.09821524504966911, 'tversky_beta': 0.9573174897435559}

Best Value: 0.24964454770088196

Best Params: {'topK': 6, 'shrink': 61, 'tversky_alpha': 0.09821524504966911, 'tversky_beta': 0.9573174897435559, 'normalize': True, 'feature_weighting': 'none'}